In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["OMP_NUM_THREADS"] = "1"

import scgpt
print("scgpt import OK")
print(scgpt.__file__)


/Users/zhuqin/miniconda3/envs/cmml3_scgpt/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


scgpt import OK
/Users/zhuqin/scGPT/scgpt/__init__.py


In [6]:
from pathlib import Path
import scanpy as sc
import pandas as pd
import os

print("cwd:", os.getcwd())

PROJECT_DIR = Path(".")
DATA_DIR = PROJECT_DIR / "data" / "processed"
OUT_DIR = DATA_DIR / "scgpt_input"
OUT_DIR.mkdir(parents=True, exist_ok=True)

REF_PATH = DATA_DIR / "pancreas_ref_counts_endocrine4_sharedgenes.h5ad"
CONTROL_PATH = DATA_DIR / "pancreas_query_control_counts_endocrine4_sharedgenes.h5ad"
AAB_PATH = DATA_DIR / "pancreas_query_aab_counts_endocrine4_sharedgenes.h5ad"
T1D_PATH = DATA_DIR / "pancreas_query_t1d_counts_endocrine4_sharedgenes.h5ad"

for p in [REF_PATH, CONTROL_PATH, AAB_PATH, T1D_PATH]:
    print(p.name, "->", p.exists(), "|", p)


cwd: /Users/zhuqin/Desktop/CMML/ICA2
pancreas_ref_counts_endocrine4_sharedgenes.h5ad -> True | data/processed/pancreas_ref_counts_endocrine4_sharedgenes.h5ad
pancreas_query_control_counts_endocrine4_sharedgenes.h5ad -> True | data/processed/pancreas_query_control_counts_endocrine4_sharedgenes.h5ad
pancreas_query_aab_counts_endocrine4_sharedgenes.h5ad -> True | data/processed/pancreas_query_aab_counts_endocrine4_sharedgenes.h5ad
pancreas_query_t1d_counts_endocrine4_sharedgenes.h5ad -> True | data/processed/pancreas_query_t1d_counts_endocrine4_sharedgenes.h5ad


In [7]:
def prepare_scgpt_adata(in_path, out_path):
    adata = sc.read_h5ad(in_path)

    if "gene_symbol" not in adata.var.columns:
        raise ValueError("gene_symbol column not found in adata.var")

    # 保留非空 gene symbol
    gene_symbol = adata.var["gene_symbol"].astype(str)
    valid = (~gene_symbol.isna()) & (gene_symbol != "") & (gene_symbol != "nan")
    adata = adata[:, valid].copy()

    # 去掉重复 gene symbol
    adata.var["gene_symbol"] = adata.var["gene_symbol"].astype(str)
    keep = ~adata.var["gene_symbol"].duplicated()
    adata = adata[:, keep].copy()

    # 改成 gene symbol 作为 var_names
    original_ids = adata.var_names.astype(str)
    adata.var_names = adata.var["gene_symbol"].astype(str)
    adata.var["ensembl_id"] = original_ids

    print(out_path.name, adata.shape)
    print(adata.obs["label"].value_counts())

    adata.write_h5ad(out_path, compression="gzip")
    return adata


In [8]:
adata_ref_scgpt = prepare_scgpt_adata(
    REF_PATH, OUT_DIR / "pancreas_ref_scgpt_input.h5ad"
)

adata_control_scgpt = prepare_scgpt_adata(
    CONTROL_PATH, OUT_DIR / "pancreas_control_scgpt_input.h5ad"
)

adata_aab_scgpt = prepare_scgpt_adata(
    AAB_PATH, OUT_DIR / "pancreas_aab_scgpt_input.h5ad"
)

adata_t1d_scgpt = prepare_scgpt_adata(
    T1D_PATH, OUT_DIR / "pancreas_t1d_scgpt_input.h5ad"
)


pancreas_ref_scgpt_input.h5ad (26474, 24233)
label
beta     11923
alpha    11541
delta     2889
pp         121
Name: count, dtype: int64
pancreas_control_scgpt_input.h5ad (8669, 24233)
label
alpha    4821
beta     3158
delta     539
pp        151
Name: count, dtype: int64
pancreas_aab_scgpt_input.h5ad (13122, 24233)
label
beta     7069
alpha    5041
delta     678
pp        334
Name: count, dtype: int64
pancreas_t1d_scgpt_input.h5ad (3267, 24233)
label
alpha    1727
beta     1071
pp        276
delta     193
Name: count, dtype: int64


In [9]:
for f in sorted(OUT_DIR.glob("*.h5ad")):
    print(f.name)


pancreas_aab_scgpt_input.h5ad
pancreas_control_scgpt_input.h5ad
pancreas_ref_scgpt_input.h5ad
pancreas_t1d_scgpt_input.h5ad


In [10]:
from scgpt.tasks import embed_data
import inspect
print(inspect.signature(embed_data))


/Users/zhuqin/miniconda3/envs/cmml3_scgpt/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


(adata_or_file: Union[anndata._core.anndata.AnnData, str, os.PathLike], model_dir: Union[str, os.PathLike], gene_col: str = 'feature_name', max_length=1200, batch_size=64, obs_to_save: Optional[list] = None, device: Union[str, torch.device] = 'cuda', use_fast_transformer: bool = True, return_new_adata: bool = False) -> anndata._core.anndata.AnnData


In [14]:
from pathlib import Path

MODEL_DIR = Path("models/scgpt_human")

print("MODEL_DIR exists:", MODEL_DIR.exists())
for f in sorted(MODEL_DIR.iterdir()):
    print(f.name)


MODEL_DIR exists: True
.DS_Store
args.json
best_model.pt
vocab.json


In [15]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["OMP_NUM_THREADS"] = "1"


In [16]:
from pathlib import Path
import scanpy as sc
import torch
from scgpt.tasks import embed_data

PROJECT_DIR = Path(".")
DATA_DIR = PROJECT_DIR / "data" / "processed" / "scgpt_input"
MODEL_DIR = PROJECT_DIR / "models" / "scgpt_human"
RESULT_DIR = PROJECT_DIR / "results" / "scgpt"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

print("MODEL_DIR exists:", MODEL_DIR.exists())
for f in sorted(MODEL_DIR.iterdir()):
    print(f.name)

device = "mps" if torch.backends.mps.is_available() else "cpu"
print("device:", device)


MODEL_DIR exists: True
.DS_Store
args.json
best_model.pt
vocab.json
device: mps


In [17]:
adata_control_scgpt = sc.read_h5ad(DATA_DIR / "pancreas_control_scgpt_input.h5ad")
print(adata_control_scgpt)
print("var columns:", list(adata_control_scgpt.var.columns))


AnnData object with n_obs × n_vars = 8669 × 24233
    obs: 'nCount_RNA', 'nFeature_RNA', 'percent.mt', 'RNA_snn_res.1.2', 'seurat_clusters', 'disease_state', 'cell_label', 'disease_ontology_term_id', 'tissue_ontology_term_id', 'assay_ontology_term_id', 'cell_type_ontology_term_id', 'is_primary_data', 'donor_id', 'development_stage_ontology_term_id', 'sex_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'suspension_type', 'tissue_type', 'cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid', 'cell_type_original', 'label', 'dataset_name', 'dataset_role', 'cohort', 'n_counts', 'n_genes_by_counts', 'total_counts'
    var: 'vst.mean', 'vst.variance', 'vst.variance.expected', 'vst.variance.standardized', 'vst.variable', 'feature_name', 'feature_reference', 'feature_biotype', 'feature_length', 'feature_type', 'gene_symbol', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'ensembl_id'
   

In [19]:
import os
import scgpt.tasks.cell_emb as cell_emb

def _fake_sched_getaffinity(pid=0):
    return set()

os.sched_getaffinity = _fake_sched_getaffinity
cell_emb.os.sched_getaffinity = _fake_sched_getaffinity

print("Patched os.sched_getaffinity for macOS.")


Patched os.sched_getaffinity for macOS.


In [ ]:
test_adata = adata_control_scgpt[:200].copy()

test_emb = embed_data(
    test_adata,
    MODEL_DIR,
    gene_col="gene_symbol",
    batch_size=8,
    obs_to_save=["label"],
    device="cpu",
    use_fast_transformer=False,
    return_new_adata=True,
)

print(test_emb)
print("obsm keys:", list(test_emb.obsm.keys()))


In [22]:
import numpy as np

print(type(test_emb.X))
print(test_emb.X.shape)

if hasattr(test_emb.X, "toarray"):
    X_test = test_emb.X.toarray()
else:
    X_test = np.asarray(test_emb.X)

print(X_test[:3, :5])


<class 'numpy.ndarray'>
(200, 512)
[[ 0.03629534 -0.00684253  0.00285329  0.04015883  0.0099612 ]
 [ 0.04061674  0.00352044  0.018143    0.0303855   0.01899205]
 [ 0.02395567 -0.00822118 -0.00675486  0.04219778  0.00322878]]


In [23]:
from pathlib import Path
import scanpy as sc
from scgpt.tasks import embed_data

PROJECT_DIR = Path(".")
DATA_DIR = PROJECT_DIR / "data" / "processed" / "scgpt_input"
MODEL_DIR = PROJECT_DIR / "models" / "scgpt_human"
RESULT_DIR = PROJECT_DIR / "results" / "scgpt"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

OBS_TO_SAVE = ["label", "cell_type", "donor_id", "disease", "disease_state", "cohort"]

def run_scgpt_embedding(in_path, out_path, model_dir, batch_size=8, device="cpu"):
    adata = sc.read_h5ad(in_path)

    keep_obs = [c for c in OBS_TO_SAVE if c in adata.obs.columns]

    adata_emb = embed_data(
        adata,
        model_dir,
        gene_col="gene_symbol",
        batch_size=batch_size,
        obs_to_save=keep_obs,
        device=device,
        use_fast_transformer=False,
        return_new_adata=True,
    )

    print("\nSaved:", out_path.name)
    print("shape:", adata_emb.shape)
    print("obs columns:", list(adata_emb.obs.columns))

    adata_emb.write_h5ad(out_path, compression="gzip")
    return adata_emb


In [27]:
import numpy as np

def stratified_subsample_adata(adata, label_col="label", max_per_class=None, random_state=42):
    if max_per_class is None:
        raise ValueError("max_per_class must be provided as a dict")

    rng = np.random.default_rng(random_state)
    keep_idx = []

    labels = adata.obs[label_col].astype(str)

    for lab, max_n in max_per_class.items():
        idx = np.where(labels.values == lab)[0]
        if len(idx) <= max_n:
            keep_idx.extend(idx.tolist())
        else:
            keep_idx.extend(rng.choice(idx, size=max_n, replace=False).tolist())

    keep_idx = np.array(sorted(keep_idx))
    return adata[keep_idx].copy()


In [28]:
adata_ref_scgpt_sub = stratified_subsample_adata(
    adata_ref_scgpt,
    label_col="label",
    max_per_class={
        "alpha": 3000,
        "beta": 3000,
        "delta": 2889,
        "pp": 121
    },
    random_state=42
)

print(adata_ref_scgpt_sub)
print(adata_ref_scgpt_sub.obs["label"].value_counts())


AnnData object with n_obs × n_vars = 9010 × 24233
    obs: 'sample', 'louvain_anno_broad', 'louvain_anno_fine', 'donor_id', 'BMI', 'HbA1c', 'insulin_content', 'glucose_SI', 'assay_ontology_term_id', 'cell_type_ontology_term_id', 'development_stage_ontology_term_id', 'disease_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'is_primary_data', 'sex_ontology_term_id', 'tissue_ontology_term_id', 'n_genes', 'n_counts', 'mt_frac', 'size_factors', 'suspension_type', 'tissue_type', 'cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid', 'cell_type_original', 'label', 'dataset_name', 'dataset_role', 'cohort', 'n_genes_by_counts', 'total_counts'
    var: 'feature_name', 'feature_reference', 'feature_biotype', 'feature_length', 'feature_type', 'gene_symbol', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'ensembl_id'
    uns: 'batch_condition', 'citation', 'default_embedding', 'organism', '

In [29]:
adata_ref_emb = embed_data(
    adata_ref_scgpt_sub,
    MODEL_DIR,
    gene_col="gene_symbol",
    batch_size=32,
    obs_to_save=["label"],
    device="cpu",
    use_fast_transformer=False,
    return_new_adata=True,
)

print(adata_ref_emb)
print(adata_ref_emb.X.shape)


scGPT - INFO - match 19701/24233 genes in vocabulary of size 60697.


/Users/zhuqin/scGPT/scgpt/tasks/cell_emb.py:120: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), torch.cuda.amp.autocast(enabled=True):
/Users/zhuqin/miniconda3/envs/cmml3_scgpt/lib/python3.10/site-packages/torch/cuda/amp/autocast_mode.py:54: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  super().__init__(
Embedding cells:   0%|          | 0/282 [00:00<?, ?it/s]/Users/zhuqin/miniconda3/envs/cmml3_scgpt/lib/python3.10/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Embedding cells: 100%|██████████| 282/282 [1:10:47<00:00, 15.06s/it]

AnnData object with n_obs × n_vars = 9010 × 512
    obs: 'label'
(9010, 512)



/Users/zhuqin/miniconda3/envs/cmml3_scgpt/lib/python3.10/site-packages/anndata/_core/anndata.py:381: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(


In [30]:
RESULT_DIR = Path("results") / "scgpt"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

adata_ref_emb.write_h5ad(
    RESULT_DIR / "pancreas_ref_scgpt_emb_subset.h5ad",
    compression="gzip"
)


In [31]:
adata_control_emb = run_scgpt_embedding(
    DATA_DIR / "pancreas_control_scgpt_input.h5ad",
    RESULT_DIR / "pancreas_control_scgpt_emb.h5ad",
    MODEL_DIR,
    batch_size=32,
    device="cpu",
)

adata_aab_emb = run_scgpt_embedding(
    DATA_DIR / "pancreas_aab_scgpt_input.h5ad",
    RESULT_DIR / "pancreas_aab_scgpt_emb.h5ad",
    MODEL_DIR,
    batch_size=32,
    device="cpu",
)

adata_t1d_emb = run_scgpt_embedding(
    DATA_DIR / "pancreas_t1d_scgpt_input.h5ad",
    RESULT_DIR / "pancreas_t1d_scgpt_emb.h5ad",
    MODEL_DIR,
    batch_size=32,
    device="cpu",
)


scGPT - INFO - match 19701/24233 genes in vocabulary of size 60697.


/Users/zhuqin/scGPT/scgpt/tasks/cell_emb.py:120: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), torch.cuda.amp.autocast(enabled=True):
/Users/zhuqin/miniconda3/envs/cmml3_scgpt/lib/python3.10/site-packages/torch/cuda/amp/autocast_mode.py:54: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  super().__init__(
Embedding cells:   0%|          | 0/271 [00:00<?, ?it/s]/Users/zhuqin/miniconda3/envs/cmml3_scgpt/lib/python3.10/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Embedding cells: 100%|██████████| 271/271 [1:00:22<00:00, 13.37s/it]
/Users/zhuqin/miniconda3/envs/cmml3_scgpt/lib/python3.10/site-packages/anndata/_core/anndata.py:381: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.


Saved: pancreas_control_scgpt_emb.h5ad
shape: (8669, 512)
obs columns: ['label', 'cell_type', 'donor_id', 'disease', 'disease_state', 'cohort']
scGPT - INFO - match 19701/24233 genes in vocabulary of size 60697.


/Users/zhuqin/scGPT/scgpt/tasks/cell_emb.py:120: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), torch.cuda.amp.autocast(enabled=True):
/Users/zhuqin/miniconda3/envs/cmml3_scgpt/lib/python3.10/site-packages/torch/cuda/amp/autocast_mode.py:54: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  super().__init__(
Embedding cells:   0%|          | 0/411 [00:00<?, ?it/s]/Users/zhuqin/miniconda3/envs/cmml3_scgpt/lib/python3.10/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Embedding cells: 100%|██████████| 411/411 [1:29:53<00:00, 13.12s/it]
/Users/zhuqin/miniconda3/envs/cmml3_scgpt/lib/python3.10/site-packages/anndata/_core/anndata.py:381: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.


Saved: pancreas_aab_scgpt_emb.h5ad
shape: (13122, 512)
obs columns: ['label', 'cell_type', 'donor_id', 'disease', 'disease_state', 'cohort']
scGPT - INFO - match 19701/24233 genes in vocabulary of size 60697.


/Users/zhuqin/scGPT/scgpt/tasks/cell_emb.py:120: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), torch.cuda.amp.autocast(enabled=True):
/Users/zhuqin/miniconda3/envs/cmml3_scgpt/lib/python3.10/site-packages/torch/cuda/amp/autocast_mode.py:54: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  super().__init__(
Embedding cells:   0%|          | 0/103 [00:00<?, ?it/s]/Users/zhuqin/miniconda3/envs/cmml3_scgpt/lib/python3.10/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Embedding cells: 100%|██████████| 103/103 [21:58<00:00, 12.80s/it]


Saved: pancreas_t1d_scgpt_emb.h5ad
shape: (3267, 512)
obs columns: ['label', 'cell_type', 'donor_id', 'disease', 'disease_state', 'cohort']



/Users/zhuqin/miniconda3/envs/cmml3_scgpt/lib/python3.10/site-packages/anndata/_core/anndata.py:381: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(


In [32]:
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

LABEL_ORDER = ["alpha", "beta", "delta", "pp"]
RESULT_DIR = Path("results") / "scgpt"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

def get_embedding_matrix(adata):
    X = adata.X
    if hasattr(X, "toarray"):
        X = X.toarray()
    return np.asarray(X)

def evaluate_dataset(name, y_true, y_pred, out_dir):
    acc = accuracy_score(y_true, y_pred)

    p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=LABEL_ORDER, average="macro", zero_division=0
    )
    p_weighted, r_weighted, f1_weighted, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=LABEL_ORDER, average="weighted", zero_division=0
    )

    metrics_df = pd.DataFrame([{
        "dataset": name,
        "n_cells": len(y_true),
        "accuracy": acc,
        "macro_precision": p_macro,
        "macro_recall": r_macro,
        "macro_f1": f1_macro,
        "weighted_precision": p_weighted,
        "weighted_recall": r_weighted,
        "weighted_f1": f1_weighted
    }])
    metrics_df.to_csv(out_dir / f"{name}_metrics.csv", index=False)

    report = classification_report(
        y_true, y_pred, labels=LABEL_ORDER, output_dict=True, zero_division=0
    )
    pd.DataFrame(report).T.to_csv(out_dir / f"{name}_classification_report.csv")

    cm = confusion_matrix(y_true, y_pred, labels=LABEL_ORDER)
    pd.DataFrame(cm, index=LABEL_ORDER, columns=LABEL_ORDER).to_csv(
        out_dir / f"{name}_confusion_matrix.csv"
    )

    print(f"\n===== {name} =====")
    print(metrics_df.to_string(index=False))

    return metrics_df


In [33]:
ref_embed = get_embedding_matrix(adata_ref_emb)
control_embed = get_embedding_matrix(adata_control_emb)
aab_embed = get_embedding_matrix(adata_aab_emb)
t1d_embed = get_embedding_matrix(adata_t1d_emb)

y_train = adata_ref_emb.obs["label"].astype(str).values
y_control = adata_control_emb.obs["label"].astype(str).values
y_aab = adata_aab_emb.obs["label"].astype(str).values
y_t1d = adata_t1d_emb.obs["label"].astype(str).values

print("ref_embed:", ref_embed.shape)
print("control_embed:", control_embed.shape)
print("aab_embed:", aab_embed.shape)
print("t1d_embed:", t1d_embed.shape)

clf = LogisticRegression(
    max_iter=2000,
    multi_class="multinomial",
    class_weight="balanced",
    random_state=42
)

clf.fit(ref_embed, y_train)


ref_embed: (9010, 512)
control_embed: (8669, 512)
aab_embed: (13122, 512)
t1d_embed: (3267, 512)


/Users/zhuqin/miniconda3/envs/cmml3_scgpt/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,42
,solver,'lbfgs'
,max_iter,2000
,multi_class,'multinomial'


In [34]:
pred_control = clf.predict(control_embed)
pred_aab = clf.predict(aab_embed)
pred_t1d = clf.predict(t1d_embed)

all_metrics = []
all_metrics.append(evaluate_dataset("control", y_control, pred_control, RESULT_DIR))
all_metrics.append(evaluate_dataset("aab", y_aab, pred_aab, RESULT_DIR))
all_metrics.append(evaluate_dataset("t1d", y_t1d, pred_t1d, RESULT_DIR))

all_metrics_df = pd.concat(all_metrics, ignore_index=True)
all_metrics_df.to_csv(RESULT_DIR / "all_metrics_summary.csv", index=False)

print("\nSaved overall summary:")
print(all_metrics_df.to_string(index=False))



===== control =====
dataset  n_cells  accuracy  macro_precision  macro_recall  macro_f1  weighted_precision  weighted_recall  weighted_f1
control     8669  0.759488         0.513884      0.581582  0.526913            0.795148         0.759488     0.774915

===== aab =====
dataset  n_cells  accuracy  macro_precision  macro_recall  macro_f1  weighted_precision  weighted_recall  weighted_f1
    aab    13122  0.699436         0.514611      0.507624  0.466847            0.843463         0.699436     0.749293

===== t1d =====
dataset  n_cells  accuracy  macro_precision  macro_recall  macro_f1  weighted_precision  weighted_recall  weighted_f1
    t1d     3267  0.766452         0.593992      0.609819  0.596106            0.780541         0.766452     0.771921

Saved overall summary:
dataset  n_cells  accuracy  macro_precision  macro_recall  macro_f1  weighted_precision  weighted_recall  weighted_f1
control     8669  0.759488         0.513884      0.581582  0.526913            0.795148        